# Workshop 2 — Deep Sequence Model (Google Colab runner)

**Course:** Machine Learning — Semester 2026-III  
**Workshop 2:** Advanced Learning Models (Part A) — Group 5

## Overview
This notebook runs the complete Workshop 2 pipeline in order from a fresh Colab
runtime. Every step calls the scripts in `src/w2/`, so the notebook and the
repository always produce the same numbers.

| Step | Script | What it does |
|---|---|---|
| 1 | `src/w2/model.py` | Architecture smoke-test (forward pass + parameter count) |
| 2 | `src/w2/train_lstm.py` | Train LSTM, log to TensorBoard + CSV, save best checkpoint |
| 3 | `src/w2/evaluate_lstm.py` | Test-set metrics + comparison table vs. W1 baselines |
| 4 | `src/w2/test_model.py` | Verify the saved checkpoint runs inference on new data |
| 5 | `src/w2/make_figures_w2.py` | Save PNG figures for the report |

Set `RETRAIN = True` in step 2 to retrain from scratch (~5 min). By default the
checkpoints already in the repository are used.

## 0. Get the repository

Clone the repository into `cartpole-ml` and move into it. If the runtime restarts,
run this cell again.

In [ ]:
%cd /content
!rm -rf cartpole-ml
!git clone https://github.com/Tonyaa11/Cartpole.git cartpole-ml
%cd /content/cartpole-ml

## 0.1 Dependencies

Colab already includes PyTorch, NumPy, pandas, matplotlib and TensorBoard. We pin
scikit-learn to **1.6.1**, the version used for the Workshop 1 results: the
comparison with the W1 baselines (step 3 and the ROC/PR figure) refits them, and
tree ensembles can change slightly between scikit-learn versions. The versions are
printed as part of the reproducibility record.

In [ ]:
!pip install -q "scikit-learn==1.6.1"
import torch, sklearn, numpy, pandas
print("torch", torch.__version__,
      "| sklearn", sklearn.__version__,
      "| numpy", numpy.__version__,
      "| pandas", pandas.__version__)

## 1. Architecture check

Runs a forward pass on a random batch and prints the output shape and total
parameter count for the default configuration (2-layer LSTM, 64 hidden units,
MLP head → 53,569 parameters).

In [ ]:
!python src/w2/model.py

## 2. Training

Trains the LSTM on the training partition (same episode split as Workshop 1 —
never re-split). The best checkpoint (highest validation F1) is saved to
`checkpoints/workshop2/model_best.pt` together with the scaler, the window
configuration and the validation threshold.

Set `RETRAIN = True` to train from scratch (~5 min). By default the existing
checkpoint in the repository is used.

In [ ]:
RETRAIN = False  # True = train from scratch (overwrites checkpoints/workshop2 and runs/workshop2)

if RETRAIN:
    !python src/w2/train_lstm.py
else:
    print("Using the checkpoint stored in the repository (RETRAIN = False).")
    print("Set RETRAIN = True to train from scratch.")

## 2.1 TensorBoard (optional)

Training loss, validation F1 and learning-rate schedule per epoch.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/workshop2/tb

## 3. Test-set evaluation

Reloads `checkpoints/workshop2/model_best.pt` (no retraining) and evaluates on
the held-out test set. Prints:
- Accuracy, precision, recall, F1, ROC-AUC, PR-AUC and confusion matrix
- Side-by-side comparison table: W1 baselines vs. W2 deep model

In [ ]:
!python src/w2/evaluate_lstm.py

## 4. Checkpoint verification

Loads the saved checkpoint and runs inference on 10 test examples. This confirms
the artifact is self-contained (weights + scaler + threshold stored together).

In [ ]:
!python src/w2/test_model.py --n 10 --split test

## 5. Report figures

Generates five PNG files saved to `runs/workshop2/figures/`:
- `fig2-architecture.png` — model schematic
- `fig2-curves.png` — training / validation learning curves
- `fig2-confusion.png` — confusion matrix at the validation threshold
- `fig2-roc-pr.png` — ROC and PR curves (deep model vs. W1 baselines)
- `fig2-comparison.png` — bar chart comparing all models

In [ ]:
!python src/w2/make_figures_w2.py
from IPython.display import Image, display
import glob
for f in sorted(glob.glob("runs/workshop2/figures/*.png")):
    print(f)
    display(Image(f, width=900))

## 6. Summary

Prints the comparison table (W1 baselines vs. W2 deep model) from the saved
JSON so you can verify the numbers match the report.

In [ ]:
import glob, json
import pandas as pd

# Latest comparison CSV written by evaluate_lstm.py
csvs = sorted(glob.glob("runs/workshop2/comparison_*.csv"))
if csvs:
    df = pd.read_csv(csvs[-1])
    display(df.round(4))
else:
    print("No comparison CSV found — run step 3 first.")

## 7. Download results

Downloads a zip with logs, checkpoint and figures. Unzip over your local
repository copy and commit.

In [ ]:
!zip -qr workshop2_results.zip runs/workshop2 checkpoints/workshop2
from google.colab import files
files.download("workshop2_results.zip")